In [1]:
import numpy as np
import pandas as pd
import subprocess
import time
from io import StringIO
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

In [33]:
#########################
# Set these variables as per your requirement
#########################

### (treatment) ###
#hic_file_1 = "inter_1.hic"
#hic_file_1 = "subsampled.hic"
hic_file_1 = "subsampled_INT11Aux.hic"
#hic_file_1 = "inter_30.hic"
### (control) ###
# hic_file_2 = "subsampled_INT1Unt.hic"
hic_file_2 = "subsampled.hic"

#anchor_file = "cluster5_anchors.bed"
anchor_file = "simpleTSS_anchors_rmdup_addedplus.txt"

output_dir = "25_to_10kb_Genes" 

num_bins = 10 # Number of pixels to plot between each anchor, default is 10 pixels
output_prefix = "INT11Aux_Vs_INT11Unt_l2fc_opporient" # Prefix for output scaled matrices, default is meta_out
resolution, res = 500, 500 # Resolution to process the Hi-C file in bp, too lazy to go through and update all the res references so just type the value twice
straw = "straw" # Path to straw
norm = 'NONE'
#create_fig = False # Set this to create figures
#min_and_max = "0,2" # Desired min and max in the heatmap scale
#color_scale = "Reds" # Matplotlib color gradient
decay = False # Include diagonal decay
#n_clusters = 6 # How many clusters for k-means is optimized below

In [11]:
#########################
# Read in anchor file
#########################

chr_list = []
anchor_list = []
with open(anchor_file, 'r') as anchors:
    line_count = 0
    for line in anchors:
        li = line.split()
        if li[0] not in chr_list:
            chr_list.append(li[0])
            anchor_list.append([])
        
        c = chr_list.index(li[0])
        strand = str(li[-1])
        if line_count == 0:
            anchor_num = len(li[1:]) - 1
            if anchor_num < 2:
                print("Must have at least 2 anchors... exiting")
                break
            line_count += 1
            totalbinnum = num_bins * (anchor_num - 1)
        if len(li[1:]) - 1 != anchor_num:
            print("Number of columns in this line in anchor file does not match the first line")
            print(li)
            break

        if strand == "-":
            anchor_list[c].append([int(float(x) / float(res)) for x in reversed(li[1:-1])])
        else:
            anchor_list[c].append([int(float(x) / float(res)) for x in li[1:-1]])
            


In [12]:
print(chr_list)

['22', '13', '17', '11', '16', '6', '19', '12', '9', '1', '14', '3', '2', '8', '5', '21', '20', '4', '7', '10', '18', '15']


In [31]:
#########################
# estimate expecteds_1 based on chr 1
#########################

firstchr = chr_list[0]
intres = int(resolution)
largestbin=0
print("estimating expecteds using the first chromosome, this should take a few minutes")
#expecteds=[]
with subprocess.Popen([straw, 'NONE', hic_file_1, firstchr, firstchr, 'BP', str(res)], stdout=subprocess.PIPE, stderr=subprocess.PIPE) as p:
    output, errors = p.communicate()
lines = output.decode('utf-8').splitlines()
for line in lines:
    myE = line.split()
    if myE[0] == "HiC":
        continue
    if myE[0] == "WARN":
        continue
    if myE[0] == "INFO":
        continue
    if myE[0] == "WARNING:":
        continue
    if int(myE[1]) > largestbin:
        largestbin = int(myE[1])
#print(largestbin)
mynumbins = int(float(largestbin)/float(resolution))
#print(mynumbins)
totpossible=[]
for eachitem in range(mynumbins+1,1,-1):
    totpossible.append(eachitem)
expecteds_1=[0.0001]*mynumbins
with subprocess.Popen([straw, 'NONE', hic_file_1, firstchr, firstchr, 'BP', str(res)], stdout=subprocess.PIPE, stderr=subprocess.PIPE) as p:
    output, errors = p.communicate()
lines = output.decode('utf-8').splitlines()
for line in lines:
    myE = line.split()
    if myE[0] == "HiC":
        continue
    if myE[0] == "WARN":
        continue
    if myE[0] == "INFO":
        continue
    if myE[0] == "WARNING:":
        continue
    distance = int(float(int(myE[1])-int(myE[0]))/float(res))
    try:
        myscore = float(myE[2]) / float(totpossible[distance])
    except IndexError as error:
        #print(distance)
        #print(len(totpossible))
        myscore = float(myE[2])
    expecteds_1[distance] += float(myscore)
print("expecteds from straw")
print(expecteds_1[:10])


estimating expecteds using the first chromosome, this should take a few minutes
expecteds from straw
[34.06853085586384, 31.40881260724862, 5.251444027404447, 3.0896193242088867, 2.073634633670477, 1.513593477530601, 1.1697974227681824, 0.939546797907453, 0.7836209539062372, 0.6646547539838124]


In [32]:
###############
# Output expecteds_1 to a file
###############
file_path = f"{output_dir}/expecteds_fullchrom_1_500bp.txt"

with open(file_path, "w") as file:
    for value in expecteds_1:
        file.write(f"{value}\n")
        


In [35]:
###############
# Read expecteds_1 in from a file
###############
# Specify the file name
#file_path = f"{output_dir}/expecteds_1_500bp.txt"
file_path = "filled_expected_chr1_500_NONE.txt"
# file_path = "filled_expected_chr1_500_NONE.txt" 

# Read the values into a list, converting each line to a float
with open(file_path, "r") as file:
    expecteds_1 = [float(line.strip()) for line in file]

# Verify the loaded values (optional)
print(expecteds_1[:10])

[41.194442749023, 41.194442749023, 4.6876587867737, 2.9549624919891, 2.2496513724327, 1.5443402528763, 1.2204830646515, 1.1140443895544414, 1.007605714457383, 0.9011670393603243]


In [20]:
#########################
# estimate expecteds_2 based on chr 1
#########################

firstchr = chr_list[0]
intres = int(resolution)
largestbin=0
print("estimating expecteds using the first chromosome, this should take a few minutes")
#expecteds=[]
with subprocess.Popen([straw, 'NONE', hic_file_2, firstchr, firstchr, 'BP', str(res)], stdout=subprocess.PIPE, stderr=subprocess.PIPE) as p:
    output, errors = p.communicate()
lines = output.decode('utf-8').splitlines()
for line in lines:
    myE = line.split()
    if myE[0] == "HiC":
        continue
    if myE[0] == "WARN":
        continue
    if myE[0] == "INFO":
        continue
    if myE[0] == "WARNING:":
        continue
    if int(myE[1]) > largestbin:
        largestbin = int(myE[1])
#print(largestbin)
mynumbins = int(float(largestbin)/float(resolution))
#print(mynumbins)
totpossible=[]
for eachitem in range(mynumbins+1,1,-1):
    totpossible.append(eachitem)
expecteds=[0.0001]*mynumbins
with subprocess.Popen([straw, 'NONE', hic_file_2, firstchr, firstchr, 'BP', str(res)], stdout=subprocess.PIPE, stderr=subprocess.PIPE) as p:
    output, errors = p.communicate()
lines = output.decode('utf-8').splitlines()
for line in lines:
    myE = line.split()
    if myE[0] == "HiC":
        continue
    if myE[0] == "WARN":
        continue
    if myE[0] == "INFO":
        continue
    if myE[0] == "WARNING:":
        continue
    distance = int(float(int(myE[1])-int(myE[0]))/float(res))
    try:
        myscore = float(myE[2]) / float(totpossible[distance])
    except IndexError as error:
        #print(distance)
        #print(len(totpossible))
        myscore = float(myE[2])
    expecteds[distance] += float(myscore)
print("expecteds from straw")
print(expecteds[:10])

estimating expecteds using the first chromosome, this should take a few minutes
expecteds from straw
[42.5302411634763, 39.144102902145214, 6.660709135455677, 3.897693892847972, 2.608152974987, 1.9031244081955754, 1.4646429595971773, 1.179165359633066, 0.9766432358432374, 0.8279658573475053]


In [21]:
###############
# Output expecteds_2 to a file
###############
file_path = f"{output_dir}/expecteds_2_500bp.txt"

with open(file_path, "w") as file:
    for value in expecteds:
        file.write(f"{value}\n")

In [37]:
###############
# Read expecteds_2 in from a file
###############

# Specify the file name
#file_path = f"{output_dir}/expecteds_2_500bp.txt"
file_path = "filled_expected_chr1_500_NONE.txt" 
#file_path = "filled_expected_chr1_500_NONE.txt" 

# Read the values into a list, converting each line to a float
with open(file_path, "r") as file:
    expecteds_2 = [float(line.strip()) for line in file]

# Verify the loaded values (optional)
print(expecteds_2[:10])



[39.385608673096, 39.385608673096, 5.5571265220642, 3.5958368778229, 2.5272114276886, 2.198220360279084, 1.869229292869568, 1.540238225460052, 1.211247158050536, 0.88225609064102]


In [ ]:
#####################
# OLD AND BAD(ish) WAY
#####################


import numpy as np
import pandas as pd
import subprocess
import matplotlib.pyplot as plt
from io import StringIO

def process_anchor_set(anchorset, hic_file_1, hic_file_2, juicertools, norm, res, decay, expecteds_1, expecteds_2, binnum, totalbinnum):
    """Process an anchor set to extract and normalize interaction matrices from two Hi-C files, 
    compute the log2 fold change, and return the final matrix."""
    
    def extract_matrix(hic_file, expecteds):
        """Extracts a distance-normalized interaction matrix from a Hi-C file."""
        matrix = np.zeros((totalbinnum, totalbinnum))

        swap = False
        if anchorset[0] > anchorset[-1]:
            swap = True
            anchorset.reverse()

        distances = [j - i for i, j in zip(anchorset[:-1], anchorset[1:])]
        startoffset = anchorset[0]
        totaldistance = sum(distances) + 1

        if not all(d >= 1 for d in distances):
            return None  # Skip this anchor set if distances are too short

        location = f"{c}:{int(anchorset[0]*res)}:{int(anchorset[-1]*res)}"
        newresults = [[], [], []]

        # Run straw command
        with subprocess.Popen([juicertools, norm, hic_file, location, location, 'BP', str(res)], stdout=subprocess.PIPE, stderr=subprocess.PIPE) as p:
            output, errors = p.communicate()

        resultstmp = output.decode('utf-8').splitlines()

        for result in resultstmp:
            re = result.split()
            if re[0] in ["HiC", "INFO", "WARN", "WARNING:"] or re[2] == "NaN":
                continue

            left = int(float(re[0]) / float(res))
            right = int(float(re[1]) / float(res))
            dist = int(right - left)
            score = float(re[2]) if decay else (float(re[2]) + 1) / (float(expecteds[dist]) + 1)

            newresults[0].append(left)
            newresults[1].append(right)
            newresults[2].append(score)

        # Create dense matrix
        dense = np.zeros((totaldistance, totaldistance))
        for x, y, z in zip(newresults[0], newresults[1], newresults[2]):
            dense[int(x) - startoffset][int(y) - startoffset] = z
            dense[int(y) - startoffset][int(x) - startoffset] = z

        dense = np.nan_to_num(dense)  # Convert NaNs to zeros

        # Mapping each region to uniform bins
        mappings = [d / binnum for d in distances]
        bins = [[] for _ in range(totalbinnum)]

        count = 0
        for d, m, a in zip(distances, mappings, anchorset[:-1]):
            for i in range(d):
                bins[int(i / m) + (binnum * count)].append(a + i)
            count += 1

        for x in range(len(bins)):
            if not bins[x]:
                bins[x].append(bins[x - 1][0])

        # Downscale the matrix to fit the binned size
        for x in range(totalbinnum):
            for y in range(totalbinnum):
                total, count = 0, 0
                for i in bins[x]:
                    for j in bins[y]:
                        count += 1
                        try:
                            total += dense[i - startoffset][j - startoffset]
                        except:
                            print("Out of bounds", anchorset, i, j)

                average = float(total) / count if count else 0
                matrix[x, y] = average
                if swap:
                    matrix[totalbinnum - 1 - y, totalbinnum - 1 - x] = average  # Swap correctly

        return matrix

    # Get matrices for both Hi-C files
    matrix_1 = extract_matrix(hic_file_1, expecteds_1)
    matrix_2 = extract_matrix(hic_file_2, expecteds_2)

    if matrix_1 is None or matrix_2 is None:
        return None  # Skip if either matrix is invalid

    # Compute log2 fold change
    log2fc_matrix = np.log2((matrix_1 + 1) / (matrix_2 + 1))

    return log2fc_matrix

# Open file to store all log2 fold change matrices
output_file_path = f"{output_dir}/{output_prefix}_log2FC_matrices.txt"
with open(output_file_path, 'w') as outfile:
    for c in chr_list:
        for anchor_set in anchor_list[chr_list.index(c)]:
            log2fc_matrix = process_anchor_set(anchor_set, hic_file_1, hic_file_2, straw, norm, res, decay, expecteds_1, expecteds_2, int(num_bins), totalbinnum)
            
            if log2fc_matrix is None:
                continue  # Skip invalid matrices

            # Write the matrix to the output file
            str_io = StringIO()
            np.savetxt(str_io, log2fc_matrix, fmt='%f')
            outfile.write(str_io.getvalue() + "\n")

            # Plot the matrix
            plt.figure(figsize=(8, 8))
            plt.imshow(log2fc_matrix, cmap="bwr", vmin=-2, vmax=2)  # Diverging colormap
            plt.colorbar(label="log2FC (Hi-C signal)")
            plt.title(f"log2FC Hi-C: {c}")
            plt.savefig(f"{output_dir}/{c}_log2FC.png")
            plt.close()



In [38]:
import numpy as np
import pandas as pd
import subprocess
import matplotlib.pyplot as plt
from io import StringIO

def process_anchor_set(anchorset, hic_file_1, hic_file_2, juicertools, norm, res, decay, expecteds_1, expecteds_2, binnum, totalbinnum):
    """Process an anchor set to extract and normalize interaction matrices from two Hi-C files, 
    compute the log2 fold change, and return the final matrix."""
    
    def extract_matrix(hic_file, expecteds):
        """Extracts a distance-normalized interaction matrix from a Hi-C file."""
        matrix = np.zeros((totalbinnum, totalbinnum))

        swap = False
        if anchorset[0] > anchorset[-1]:
            swap = True
            anchorset.reverse()

        distances = [j - i for i, j in zip(anchorset[:-1], anchorset[1:])]
        startoffset = anchorset[0]
        totaldistance = sum(distances) + 1

        if not all(d >= 1 for d in distances):
            return None  # Skip this anchor set if distances are too short

        location = f"{c}:{int(anchorset[0]*res)}:{int(anchorset[-1]*res)}"
        newresults = [[], [], []]

        #print(f"Running straw for {hic_file} on location {location}")
        with subprocess.Popen([juicertools, norm, hic_file, location, location, 'BP', str(res)], 
                              stdout=subprocess.PIPE, stderr=subprocess.PIPE) as p:
            output, errors = p.communicate()

        #print(f"Straw errors: {errors.decode('utf-8')}")

        resultstmp = output.decode('utf-8').splitlines()

        for result in resultstmp:
            re = result.split()
            if re[0] in ["HiC", "INFO", "WARN", "WARNING:"] or re[2] == "NaN":
                continue

            left = int(float(re[0]) / float(res))
            right = int(float(re[1]) / float(res))
            dist = int(right - left)
            score = float(re[2]) if decay else (float(re[2]) + 1) / (float(expecteds[dist]) + 1)

            newresults[0].append(left)
            newresults[1].append(right)
            newresults[2].append(score)

        # Check if any values were extracted
        if len(newresults[0]) == 0:
            print(f"No valid interactions found for {hic_file} at {location}. Returning None.")
            return None

        # Create dense matrix
        dense = np.zeros((totaldistance, totaldistance))
        for x, y, z in zip(newresults[0], newresults[1], newresults[2]):
            dense[int(x) - startoffset][int(y) - startoffset] = z
            dense[int(y) - startoffset][int(x) - startoffset] = z

        dense = np.nan_to_num(dense)  # Convert NaNs to zeros

        # Mapping each region to uniform bins
        mappings = [d / binnum for d in distances]
        bins = [[] for _ in range(totalbinnum)]

        count = 0
        for d, m, a in zip(distances, mappings, anchorset[:-1]):
            for i in range(d):
                bins[int(i / m) + (binnum * count)].append(a + i)
            count += 1

        for x in range(len(bins)):
            if not bins[x]:
                bins[x].append(bins[x - 1][0])

        # Downscale the matrix to fit the binned size
        for x in range(totalbinnum):
            for y in range(totalbinnum):
                total, count = 0, 0
                for i in bins[x]:
                    for j in bins[y]:
                        count += 1
                        try:
                            total += dense[i - startoffset][j - startoffset]
                        except:
                            print("Out of bounds", anchorset, i, j)

                average = float(total) / count if count else 0
                matrix[x, y] = average
                if swap:
                    matrix[totalbinnum - 1 - y, totalbinnum - 1 - x] = average  # Swap correctly

        #print(f"Extracted matrix for {hic_file} with shape: {matrix.shape}")
        return matrix

    #print(f"Extracting matrix for: {hic_file_1}")
    matrix_1 = extract_matrix(hic_file_1, expecteds_1)

    if matrix_1 is None:
        print(f"Failed to extract Hi-C matrix for {hic_file_1}")

    #print(f"Extracting matrix for: {hic_file_2}")
    matrix_2 = extract_matrix(hic_file_2, expecteds_2)

    if matrix_2 is None:
        print(f"Failed to extract Hi-C matrix for {hic_file_2}")

    if matrix_1 is None or matrix_2 is None:
        return None

    log2fc_matrix = np.log2((matrix_1 + 1) / (matrix_2 + 1))
    return log2fc_matrix

all_matrices = []

# Process each anchor set and store matrices
for c in chr_list:
    print(f"Processing chromosome: {c}")
    for anchor_set in anchor_list[chr_list.index(c)]:
        log2fc_matrix = process_anchor_set(anchor_set, hic_file_1, hic_file_2, straw, norm, res, decay, expecteds_1, expecteds_2, int(num_bins), totalbinnum)

        if log2fc_matrix is None:
            print(f"Skipping anchor set {anchor_set} (returned None)")
            continue

        #print(f"Appending matrix, shape: {log2fc_matrix.shape}")
        all_matrices.append(log2fc_matrix)

if not all_matrices:
    print("No valid matrices were generated.")



Processing chromosome: 22
No valid interactions found for subsampled_INT11Aux.hic at 22:17603500:17663500. Returning None.
Failed to extract Hi-C matrix for subsampled_INT11Aux.hic
No valid interactions found for subsampled.hic at 22:17603500:17663500. Returning None.
Failed to extract Hi-C matrix for subsampled.hic
Skipping anchor set [35207, 35257, 35277, 35327] (returned None)
No valid interactions found for subsampled_INT11Aux.hic at 22:20470500:20532500. Returning None.
Failed to extract Hi-C matrix for subsampled_INT11Aux.hic
No valid interactions found for subsampled.hic at 22:20470500:20532500. Returning None.
Failed to extract Hi-C matrix for subsampled.hic
Skipping anchor set [40941, 40991, 41015, 41065] (returned None)
No valid interactions found for subsampled_INT11Aux.hic at 22:20092000:20156500. Returning None.
Failed to extract Hi-C matrix for subsampled_INT11Aux.hic
No valid interactions found for subsampled.hic at 22:20092000:20156500. Returning None.
Failed to extract

Processing chromosome: 17
Processing chromosome: 11
Processing chromosome: 16
Processing chromosome: 6
Processing chromosome: 19
Processing chromosome: 12
Processing chromosome: 9
Processing chromosome: 1
Processing chromosome: 14
Processing chromosome: 3
Processing chromosome: 2
Processing chromosome: 8
Processing chromosome: 5
Processing chromosome: 21
Processing chromosome: 20
Processing chromosome: 4
Processing chromosome: 7
Processing chromosome: 10
Processing chromosome: 18
Processing chromosome: 15


In [39]:

# Compute average and median matrices
#if all_matrices:
#everything below this should be indented when running in full script.
all_matrices = np.array(all_matrices)  # Convert list to numpy array

avg_matrix = np.mean(all_matrices, axis=0)
median_matrix = np.median(all_matrices, axis=0)

# Define output filenames
avg_matrix_path = f"{output_dir}/{output_prefix}_{norm}_average_log2FC_matrix.txt"
median_matrix_path = f"{output_dir}/{output_prefix}_{norm}_median_log2FC_matrix.txt"

# Save as .txt files
np.savetxt(avg_matrix_path, avg_matrix, fmt='%f')
np.savetxt(median_matrix_path, median_matrix, fmt='%f')

print(f"Saved average matrix: {avg_matrix_path}")
print(f"Saved median matrix: {median_matrix_path}")

# Plot average matrix
plt.figure(figsize=(8, 8))
plt.imshow(avg_matrix, cmap="bwr", vmin=-0.25, vmax=0.25)
plt.colorbar(label="log2FC (Hi-C signal)")
plt.title("Average log2FC Hi-C Matrix")
plt.savefig(f"{output_dir}/{output_prefix}_{norm}_average_log2FC.png")
plt.close()

# Plot median matrix
plt.figure(figsize=(8, 8))
plt.imshow(median_matrix, cmap="bwr", vmin=-0.25, vmax=0.25)
plt.colorbar(label="log2FC (Hi-C signal)")
plt.title("Median log2FC Hi-C Matrix")
plt.savefig(f"{output_dir}/{output_prefix}_{norm}_median_log2FC.png")
plt.close()

print("Generated and saved average and median matrices.")

#else:
    #print("No valid matrices were generated.")

Saved average matrix: INT11Aux_Vs_INT11Unt_l2fc_opporient_NONE_average_log2FC_matrix.txt
Saved median matrix: INT11Aux_Vs_INT11Unt_l2fc_opporient_NONE_median_log2FC_matrix.txt
Generated and saved average and median matrices.


In [4]:
################
# Plot from file
################

input_txt_path = "INT11_L2FC_NearbyOpposites_average_log2FC_matrix.txt"
title = "Fella"
output_image_path = "INT11_L2FC_NearbyOpposites_average_log2FC_1.png"

matrix = np.loadtxt(input_txt_path)

plt.figure(figsize=(8, 8))
plt.imshow(matrix, cmap="bwr", vmin=-1, vmax=1)
plt.colorbar(label="log2FC (Hi-C signal)")
plt.title(title)
plt.savefig(output_image_path)
plt.close()



In [ ]:
log2fc_matrix = process_anchor_set(anchor_set, hic_file_1, hic_file_2, straw, res, decay, expecteds_1, expecteds_2, int(num_bins), totalbinnum)

if log2fc_matrix is None:
    print(f"Skipping anchor set {anchor_set} (returned None)")  # Debugging print
    continue

In [ ]:
import os
print(f"Hi-C File 1 exists? {os.path.exists(hic_file_1)}")
print(f"Hi-C File 2 exists? {os.path.exists(hic_file_2)}")